In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, make_scorer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

import pickle

In [2]:
target_train = pd.read_parquet("../data/target_train.parquet")
network_train = pd.read_parquet("../data/network_train.parquet")
weather_train = pd.read_parquet("../data/weather_train.parquet")
weather_test = pd.read_parquet("../data/weather_test.parquet")
network_test = pd.read_parquet("../data/network_test.parquet")

In [3]:


### functions already used for EDA

def flatten_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ["_".join([str(i) for i in col]) for col in df.columns]
    return df


def interpolate_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_index()
    df = df.interpolate(method="time")
    df = df.ffill()
    return df


def aggregate_weather(df: pd.DataFrame) -> pd.DataFrame:
    ssrd_cols = [c for c in df.columns if c.endswith("ssrd")]
    tcc_cols  = [c for c in df.columns if c.endswith("tcc")]
    temp_cols = [c for c in df.columns if c.endswith("2t")]
    wind_cols = [c for c in df.columns if c.endswith("100ws")]

    flat = pd.DataFrame(index=df.index)
    flat["ssrd_mean"] = df[ssrd_cols].mean(axis=1)
    flat["ssrd_std"]  = df[ssrd_cols].std(axis=1)
    flat["tcc_mean"]  = df[tcc_cols].mean(axis=1)
    flat["tcc_std"]   = df[tcc_cols].std(axis=1)
    flat["temp_mean"] = df[temp_cols].mean(axis=1)
    flat["temp_std"] = df[temp_cols].std(axis=1)
    flat["wind_mean"] = df[wind_cols].mean(axis=1)
    flat["wind_std"]  = df[wind_cols].std(axis=1)

    return flat.bfill().ffill()


def prepare_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multi-index columns, interpolate missing values, and aggregate to per-hour stats."""
    df = flatten_weather_columns(df)
    df = interpolate_weather(df)
    df = aggregate_weather(df)
    df = df.reset_index().rename(columns={df.index.name or "index": "ds"})
    return df




def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tcc_roll_6h_std"]  = df["tcc_mean"].rolling(6).std()
    # Add new rolling features for tcc_mean, temp_mean, and wind_mean
    df["tcc_roll_3h_mean"] = df["tcc_mean"].rolling(3).mean()
    df["tcc_roll_24h_mean"] = df["tcc_mean"].rolling(24).mean()
    df["temp_roll_3h_mean"] = df["temp_mean"].rolling(3).mean()
    df["temp_roll_24h_mean"] = df["temp_mean"].rolling(24).mean()
    df["wind_roll_3h_mean"] = df["wind_mean"].rolling(3).mean()
    df["wind_roll_24h_mean"] = df["wind_mean"].rolling(24).mean()
    df["temp_roll_6h_std"]  = df["temp_mean"].rolling(6).std()
    df["wind_roll_6h_std"]  = df["wind_mean"].rolling(6).std()

    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(df["ds"])
    df["hour"]      = dt.dt.hour
    df["month"]     = dt.dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df



def engineer_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add rolling stats, time encodings, and solar features, then drop NaN rows."""

    df = add_rolling_features(df)
    df = add_time_features(df)

    return df.dropna()

def prepare_wind_data(target_path, weather_path, network_path):
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    wind_df = target_train[['FR_wind_actual']].join(weather_features)
    wind_df = wind_df.join(network_train[['FR_capacity_wind']])

    return wind_df.dropna()

def prepare_load_data(target_path: str, weather_path: str, network_path: str) -> pd.DataFrame:
    """Complete pipeline for load (includes network features)."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    load_df = target_train[['FR_load_actual']].join(weather_features).join(network_train)

    return load_df
##for train test split
def split_data_chronologically(df: pd.DataFrame, target_col: str, test_size: float = 0.2):
    split_point = int(len(df) * (1 - test_size))
    train = df.iloc[:split_point]
    valid = df.iloc[split_point:]

    X_train = train.drop(target_col, axis=1)
    y_train = train[target_col]
    X_valid = valid.drop(target_col, axis=1)
    y_valid = valid[target_col]

    print('Training set shape:', train.shape)
    print('Validation set shape:', valid.shape)

    return X_train, y_train, X_valid, y_valid


##for evaluation
def wmape(y_true, y_pred):
    """Weighted Mean Absolute Percentage Error"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

class TimeSeriesEvaluator:
    def compute_standard_metrics(self):
        return {
            'Target': self.target_name,
            'RMSE': np.sqrt(mean_squared_error(self.y_true, self.y_pred)),
            'MAE': mean_absolute_error(self.y_true, self.y_pred),
            'WMAPE': wmape(self.y_true, self.y_pred),
            'R²': r2_score(self.y_true, self.y_pred),
        }

In [4]:
weather_flat = prepare_weather(weather_train)
weather_flat.head()

,ds,ssrd_mean,ssrd_std,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std
0,2020-01-01 00:00:00+00:00,0.0,0.0,0.490648,0.425557,3.114028,3.679529,4.431250,2.351552
1,2020-01-01 01:00:00+00:00,0.0,0.0,0.499319,0.416623,2.807910,3.637566,4.399936,2.316953
2,2020-01-01 02:00:00+00:00,0.0,0.0,0.518816,0.414566,2.749704,3.592852,4.380681,2.291575
3,2020-01-01 03:00:00+00:00,0.0,0.0,0.534694,0.409608,2.656238,3.711994,4.475877,2.365991
4,2020-01-01 04:00:00+00:00,0.0,0.0,0.548571,0.407396,2.599818,3.779075,4.508040,2.380406


In [5]:
# Re-apply prepare_weather to get a clean DataFrame (with 'ds' as a column) as input for engineer_weather_features.
weather_flat = prepare_weather(weather_train)

# Now, apply the comprehensive feature engineering function
weather_flat = engineer_weather_features(weather_flat)

# Set 'ds' as the index for consistency with subsequent join operations, and remove the original 'ds' column.
weather_flat = weather_flat.set_index('ds')

print("Engineered weather_flat DataFrame head:")
display(weather_flat.head())
print("\nEngineered weather_flat DataFrame info:")
weather_flat.info()

Engineered weather_flat DataFrame head:


,ssrd_mean,ssrd_std,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,tcc_roll_6h_std,tcc_roll_3h_mean,...,wind_roll_3h_mean,wind_roll_24h_mean,temp_roll_6h_std,wind_roll_6h_std,hour,month,hour_sin,hour_cos,month_sin,month_cos
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01 23:00:00+00:00,0.0,0.0,0.568608,0.425440,3.889279,3.470744,3.737905,2.015928,0.028905,0.571068,...,3.681870,3.828715,0.280911,0.071246,23,1,-0.258819,0.965926,0.5,0.866025
2020-01-02 00:00:00+00:00,0.0,0.0,0.591098,0.397544,4.024190,3.424526,3.861214,2.025915,0.019537,0.575693,...,3.762191,3.804964,0.148966,0.083488,0,1,0.000000,1.000000,0.5,0.866025
2020-01-02 01:00:00+00:00,0.0,0.0,0.626592,0.391434,3.880080,3.565624,4.081434,2.093133,0.021942,0.595433,...,3.893518,3.791693,0.102429,0.171253,1,1,0.258819,0.965926,0.5,0.866025
2020-01-02 02:00:00+00:00,0.0,0.0,0.647933,0.395149,3.885397,3.561708,4.182189,2.126657,0.033401,0.621874,...,4.041613,3.783423,0.076456,0.225824,2,1,0.500000,0.866025,0.5,0.866025
2020-01-02 03:00:00+00:00,0.0,0.0,0.652530,0.402927,3.899515,3.605973,4.303050,2.085833,0.038484,0.642352,...,4.188891,3.776222,0.059011,0.250485,3,1,0.707107,0.707107,0.5,0.866025



Engineered weather_flat DataFrame info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 43826 entries, 2020-01-01 23:00:00+00:00 to 2025-01-01 00:00:00+00:00
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ssrd_mean           43826 non-null  float32
 1   ssrd_std            43826 non-null  float32
 2   tcc_mean            43826 non-null  float32
 3   tcc_std             43826 non-null  float32
 4   temp_mean           43826 non-null  float32
 5   temp_std            43826 non-null  float32
 6   wind_mean           43826 non-null  float32
 7   wind_std            43826 non-null  float32
 8   tcc_roll_6h_std     43826 non-null  float64
 9   tcc_roll_3h_mean    43826 non-null  float64
 10  tcc_roll_24h_mean   43826 non-null  float64
 11  temp_roll_3h_mean   43826 non-null  float64
 12  temp_roll_24h_mean  43826 non-null  float64
 13  wind_roll_3h_mean   43826 non-null  float64
 14  wind_roll_24h_

In [6]:
 # Drop original 'hour' and 'month' columns after creating cyclical features
weather_flat = weather_flat.drop(columns=['hour', 'month'])

In [7]:
weather_flat ['ssrd_log'] = np.log1p(weather_flat ['ssrd_mean'])
weather_flat ['ssrd_std_log'] = np.log1p(weather_flat ['ssrd_std'])

c:\Users\cherr\anaconda3\envs\ml\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [8]:
weather_flat = weather_flat.drop(columns=['ssrd_mean', 'ssrd_std'])

CREATE WIND DATAFRAME


In [9]:

wind_df = weather_flat.join(network_train[['FR_capacity_wind']])
wind_df.dropna()



,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,tcc_roll_6h_std,tcc_roll_3h_mean,tcc_roll_24h_mean,temp_roll_3h_mean,...,wind_roll_24h_mean,temp_roll_6h_std,wind_roll_6h_std,hour_sin,hour_cos,month_sin,month_cos,ssrd_log,ssrd_std_log,FR_capacity_wind
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01 23:00:00+00:00,0.568608,0.425440,3.889279,3.470744,3.737905,2.015928,0.028905,0.571068,0.588893,3.970811,...,3.828715,0.280911,0.071246,-0.258819,0.965926,5.000000e-01,0.866025,0.0,0.0,49739.039062
2020-01-02 00:00:00+00:00,0.591098,0.397544,4.024190,3.424526,3.861214,2.025915,0.019537,0.575693,0.593079,3.961289,...,3.804964,0.148966,0.083488,0.000000,1.000000,5.000000e-01,0.866025,0.0,0.0,49739.250000
2020-01-02 01:00:00+00:00,0.626592,0.391434,3.880080,3.565624,4.081434,2.093133,0.021942,0.595433,0.598382,3.931183,...,3.791693,0.102429,0.171253,0.258819,0.965926,5.000000e-01,0.866025,0.0,0.0,49739.460938
2020-01-02 02:00:00+00:00,0.647933,0.395149,3.885397,3.561708,4.182189,2.126657,0.033401,0.621874,0.603762,3.929889,...,3.783423,0.076456,0.225824,0.500000,0.866025,5.000000e-01,0.866025,0.0,0.0,49739.703125
2020-01-02 03:00:00+00:00,0.652530,0.402927,3.899515,3.605973,4.303050,2.085833,0.038484,0.642352,0.608672,3.888331,...,3.776222,0.059011,0.250485,0.707107,0.707107,5.000000e-01,0.866025,0.0,0.0,49739.914062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 20:00:00+00:00,0.423017,0.451998,3.870347,3.879590,6.656932,3.539675,0.023896,0.426471,0.622750,4.173524,...,4.578412,1.000897,0.509063,-0.866025,0.500000,-2.449294e-16,1.000000,0.0,0.0,74666.875000
2024-12-31 21:00:00+00:00,0.419937,0.449693,3.623478,3.952355,6.903283,3.531578,0.010857,0.423616,0.616099,3.886345,...,4.749639,0.681524,0.457719,-0.707107,0.707107,-2.449294e-16,1.000000,0.0,0.0,74667.304688
2024-12-31 22:00:00+00:00,0.442860,0.455166,3.388112,3.972077,7.160623,3.558408,0.007914,0.428605,0.609926,3.627312,...,4.926148,0.546603,0.454808,-0.500000,0.866025,-2.449294e-16,1.000000,0.0,0.0,74667.750000


In [10]:

target_train['FR_wind_actual']= (
    target_train['FR_wind_actual']
    .interpolate(method="time")
    .bfill()
    .ffill()
    .clip(lower=0)
)

y=target_train[['FR_wind_actual']]

In [11]:
wind_df = target_train[['FR_wind_actual']].join(wind_df)
wind_df = wind_df.dropna()

In [12]:
wind_df.columns

Index(['FR_wind_actual', 'tcc_mean', 'tcc_std', 'temp_mean', 'temp_std',
       'wind_mean', 'wind_std', 'tcc_roll_6h_std', 'tcc_roll_3h_mean',
       'tcc_roll_24h_mean', 'temp_roll_3h_mean', 'temp_roll_24h_mean',
       'wind_roll_3h_mean', 'wind_roll_24h_mean', 'temp_roll_6h_std',
       'wind_roll_6h_std', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
       'ssrd_log', 'ssrd_std_log', 'FR_capacity_wind'],
      dtype='object')

Split data

In [14]:
X_train, y_train, X_valid, y_valid = split_data_chronologically(wind_df, target_col='FR_wind_actual',test_size=0.2)


Training set shape: (34640, 23)
Validation set shape: (8661, 23)


In [15]:
X_train.to_parquet('../data/X_train_wind.parquet')
X_valid.to_parquet('../data/X_valid_wind.parquet')
y_train.to_frame().to_parquet('../data/y_train_wind.parquet')
y_valid.to_frame().to_parquet('../data/y_valid_wind.parquet')
print("Wind features saved.")

Wind features saved.


 # TRAIN MODEL

## 1. LGBM

In [ ]:


wind_model = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

wind_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse"
)




[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013800 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4652
[LightGBM] [Info] Number of data points in the train set: 34640, number of used features: 22
[LightGBM] [Info] Start training from score 4498.247004


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
              num_leaves=64, objective='regression', random_state=42,
              subsample=0.8)

EVALUATION

In [ ]:

y_pred_train_lgbm = wind_model.predict(X_train)

mae_train_lgbm = mean_absolute_error(y_train, y_pred_train_lgbm)
rmse_train_lgbm = np.sqrt(mean_squared_error(y_train, y_pred_train_lgbm))
wmape_train_lgbm  = np.sum(np.abs(y_train - y_pred_train_lgbm)) / np.sum(np.abs(y_train))
r2_train_lgbm = r2_score(y_train, y_pred_train_lgbm)

print("LightGBM Model Performance on Training Data:")
print(f"MAE: {mae_train_lgbm}")
print(f"RMSE: {rmse_train_lgbm}")
print(f"WMAPE: {wmape_train_lgbm :.4f}")
print(f"R²: {r2_train_lgbm}")


LightGBM Model Performance on Training Data:
MAE: 154.73079442992744
RMSE: 233.17048234514675
WMAPE: 0.0344
R²: 0.9949445018711811


In [ ]:
y_pred = wind_model.predict(X_valid)

mae_train_lgbm = mean_absolute_error(y_valid, y_pred)
rmse_train_lgbm = np.sqrt(mean_squared_error(y_valid, y_pred))
r2_train_lgbm = r2_score(y_valid, y_pred)
wmape = np.sum(np.abs(y_valid - y_pred)) / np.sum(np.abs(y_valid))
print("LightGBM Model Performance on Test Data:")
print(f"MAE: {mae_train_lgbm}")
print(f"RMSE: {rmse_train_lgbm}")
print(f"WMAPE: {wmape:.4f}")
print(f"R²: {r2_train_lgbm}")


LightGBM Model Performance on Test Data:
MAE: 874.9527199194773
RMSE: 1290.1322997147702
WMAPE: 0.1657
R²: 0.894215851724379


Overfitting!

### Hyperparameter Tuning for LightGBM

In [16]:
def wmape(y_true, y_pred):
    """Weighted Mean Absolute Percentage Error"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

param_dist = {
    'n_estimators': randint(500, 2000), # Number of boosting rounds
    'learning_rate': uniform(0.01, 0.2), # Step size shrinkage to prevent overfitting
    'num_leaves': randint(20, 80), # Max tree leaves for base learners
    'max_depth': randint(5, 15), # Max tree depth for base learners
    'min_child_samples': randint(20, 100), # Minimum number of data needed in a child (leaf) to split
    'subsample': uniform(0.6, 0.4), # Subsample ratio of the training instance
    'colsample_bytree': uniform(0.6, 0.4) # Subsample ratio of columns when constructing each tree
}

# Define the custom WMAPE scorer for RandomizedSearchCV
wmape_scorer = make_scorer(wmape, greater_is_better=False) # 'greater_is_better=False' since lower WMAPE is better

tscv = TimeSeriesSplit(n_splits=3) # Keeping 3 splits as in your executed cell

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=LGBMRegressor(objective='regression', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50, # Number of parameter settings that are sampled
    cv=tscv,
    scoring=wmape_scorer, # Using the custom WMAPE scorer
    verbose=2,
    random_state=42,
    n_jobs=-1 # Use all available cores
)

# Fit RandomizedSearchCV to the training data
print("Starting RandomizedSearchCV for LGBM...")
random_search.fit(X_train, y_train)
print("RandomizedSearchCV for LGBM finished.")

# Get the best parameters and the best model
best_params_lgbm = random_search.best_params_
best_model_lgbm = random_search.best_estimator_

print("Best parameters found for LGBM:", best_params_lgbm)

Starting RandomizedSearchCV for LGBM...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008531 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4652
[LightGBM] [Info] Number of data points in the train set: 34640, number of used features: 22
[LightGBM] [Info] Start training from score 4498.247004
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

### Evaluate the best LGBM model after tuning

In [17]:
# Make predictions on the validation set using the best model
y_pred_tuned_lgbm = best_model_lgbm.predict(X_valid)

# Calculate metrics for the tuned LGBM model
mae_tuned_lgbm = mean_absolute_error(y_valid, y_pred_tuned_lgbm)
rmse_tuned_lgbm = np.sqrt(mean_squared_error(y_valid, y_pred_tuned_lgbm))
wmape_tuned_lgbm = wmape(y_valid, y_pred_tuned_lgbm)
r2_tuned_lgbm = r2_score(y_valid, y_pred_tuned_lgbm)

print("LightGBM Tuned Model Performance on Validation Data:")
print(f"MAE: {mae_tuned_lgbm}")
print(f"RMSE: {rmse_tuned_lgbm}")
print(f"WMAPE: {wmape_tuned_lgbm:.4f}")
print(f"R²: {r2_tuned_lgbm}")

LightGBM Tuned Model Performance on Validation Data:
MAE: 843.9191357001858
RMSE: 1270.6080840962388
WMAPE: 0.1598
R²: 0.8973933933074565


Even after hyperparameter tuning, the LightGBM model still exhibits signs of overfitting.
R²: 0.8974 implies that approximately 89.74% of the variability in the actual wind data on the validation set is explained by this tuned LightGBM model.

## 2. XGBoost

In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
y_pred_train_xgb = xgb_model.predict(X_train)

mae_train_xgb = mean_absolute_error(y_train, y_pred_train_xgb)
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))
wmape_train_xgb  = np.sum(np.abs(y_train - y_pred_train_xgb)) / np.sum(np.abs(y_train))
r2_train_xgb = r2_score(y_train, y_pred_train_xgb)

print("XGBoost Model Performance on Training Data:")
print(f"MAE: {mae_train_xgb}")
print(f"RMSE: {rmse_train_xgb}")
print(f"WMAPE: {wmape_train_xgb :.4f}")
print(f"R²: {r2_train_xgb}")

XGBoost Model Performance on Training Data:
MAE: 202.8517303466797
RMSE: 276.1042460511971
WMAPE: 0.0451
R²: 0.9929113388061523


In [ ]:
y_pred_xgb = xgb_model.predict(X_valid)

mae_valid_xgb = mean_absolute_error(y_valid, y_pred_xgb)
rmse_valid_xgb = np.sqrt(mean_squared_error(y_valid, y_pred_xgb))
wmape_valid_xgb = np.sum(np.abs(y_valid - y_pred_xgb)) / np.sum(np.abs(y_valid))
r2_valid_xgb = r2_score(y_valid, y_pred_xgb)

print("XGBoost Model Performance on Validation Data:")
print(f"MAE: {mae_valid_xgb}")
print(f"RMSE: {rmse_valid_xgb}")
print(f"WMAPE: {wmape_valid_xgb:.4f}")
print(f"R²: {r2_valid_xgb}")

XGBoost Model Performance on Validation Data:
MAE: 997.1944580078125
RMSE: 1363.5825057546024
WMAPE: 0.1888
R²: 0.8818278908729553




Both models exhibit signs of overfitting, as their performance on the training data is much better than on the validation data.

Based on the above
 metrics, the **Tuned LightGBM model shows better performance** on the validation set compared to the XGBoost model across all evaluated metrics:

*   **Lower MAE**: LightGBM has a smaller average absolute error.
*   **Lower RMSE**: LightGBM has a smaller root mean squared error, indicating fewer large errors.
*   **Lower WMAPE**: LightGBM has a smaller weighted mean absolute percentage error, suggesting better relative accuracy.
*   **Higher R²**: LightGBM explains a larger proportion of the variance in the target variable.

This indicates that the hyperparameter tuning for LightGBM was effective in improving its generalization capabilities over the default XGBoost model parameters

In [18]:
with open("tuned_lgbm_wind_model.pkl", "wb") as f:
    pickle.dump(best_model_lgbm, f)

print("Model saved to tuned_lgbm_wind_model.pkl")

Model saved to tuned_lgbm_wind_model.pkl


# Predict on test data

In [19]:
network_test=network_test[['FR_capacity_wind']]

In [20]:
weather_test_flat = prepare_weather(weather_test)
weather_test_flat = engineer_weather_features(weather_test_flat)
weather_test_flat = weather_test_flat.set_index('ds')
weather_test_flat = weather_test_flat.drop(columns=['hour', 'month'])

weather_test_flat['ssrd_log'] = np.log1p(weather_test_flat['ssrd_mean'])
weather_test_flat['ssrd_std_log'] = np.log1p(weather_test_flat['ssrd_std'])
weather_test_flat = weather_test_flat.drop(columns=['ssrd_mean', 'ssrd_std'])

X_test = weather_test_flat.join(network_test, how='inner')

print(X_test.shape)
X_test.head()

(8737, 22)


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,tcc_roll_6h_std,tcc_roll_3h_mean,tcc_roll_24h_mean,temp_roll_3h_mean,...,wind_roll_24h_mean,temp_roll_6h_std,wind_roll_6h_std,hour_sin,hour_cos,month_sin,month_cos,ssrd_log,ssrd_std_log,FR_capacity_wind
2025-01-01 23:00:00+00:00,0.922994,0.223290,6.066291,2.983555,10.200329,3.912246,0.028232,0.919050,0.701675,6.195443,...,9.648611,0.427847,0.257023,-0.258819,0.965926,0.5,0.866025,0.0,0.0,74678.609375
2025-01-02 00:00:00+00:00,0.901928,0.236845,6.123503,2.876559,9.950920,4.013051,0.016299,0.915792,0.711806,6.107876,...,9.740503,0.313238,0.307684,0.000000,1.000000,0.5,0.866025,0.0,0.0,74679.062500
2025-01-02 01:00:00+00:00,0.905479,0.229249,6.088181,2.853961,9.728351,4.218218,0.009746,0.910133,0.722189,6.092659,...,9.810652,0.216940,0.359405,0.258819,0.965926,0.5,0.866025,0.0,0.0,74679.507812
2025-01-02 02:00:00+00:00,0.890294,0.241391,5.914766,2.921684,9.379213,4.408255,0.012621,0.899233,0.732035,6.042150,...,9.856210,0.153021,0.430592,0.500000,0.866025,0.5,0.866025,0.0,0.0,74679.921875
2025-01-02 03:00:00+00:00,0.902722,0.231171,5.666475,2.897085,9.146226,4.524261,0.012788,0.899498,0.742244,5.889808,...,9.878407,0.181014,0.468451,0.707107,0.707107,0.5,0.866025,0.0,0.0,74680.351562


In [23]:
y_pred_test = best_model_lgbm.predict(X_test)
y_pred_test = pd.Series(y_pred_test, index=X_test.index, name="FR_wind_predicted")
print(y_pred_test.shape)
y_pred_test.head()

(8737,)


,FR_wind_predicted
2025-01-01 23:00:00+00:00,14589.801745
2025-01-02 00:00:00+00:00,14108.912375
2025-01-02 01:00:00+00:00,13732.403383
2025-01-02 02:00:00+00:00,12876.911681
2025-01-02 03:00:00+00:00,12261.214130


In [ ]:

os.makedirs("../predictions", exist_ok=True)

w_predictions = y_pred_test.reset_index()
w_predictions.columns = ["datetime", "FR_wind_predicted"]
w_predictions.to_parquet("../predictions/FR_wind_predicted_2025.parquet", index=False)


w_predictions.head()

(8737, 2)


,datetime,FR_wind_predicted
0,2025-01-01 23:00:00+00:00,14589.801745
1,2025-01-02 00:00:00+00:00,14108.912375
2,2025-01-02 01:00:00+00:00,13732.403383
3,2025-01-02 02:00:00+00:00,12876.911681
4,2025-01-02 03:00:00+00:00,12261.214130
